# Lesson 2: One tool

**Project:** Learn-AI-Agent, building toward a work log agent
**Model:** `qwen3:8b`, running on this PC through Ollama

## Where lesson 1 left off

Lesson 1 ended with the model writing `LOGGED: Cornerpost Watch | Fixed the alert email bug`. That looked like a record, but nothing was saved. It also turned "cleaning the meat price data" into a project called "Meat Price Data", a name it made up.

## What we are looking for

By the end of this notebook you should be able to say yes to five things:

1. I can write a tool: an ordinary Python function that does one job.
2. I can describe that tool to the model so it knows when and how to ask for it.
3. I have seen that the model never runs the tool. It asks, and my code runs it.
4. I can send the tool's result back so the model can finish its answer.
5. I can measure how often the model asks for the tool correctly.

This is level 2 from our chat: fixed steps with one model decision in the middle. It is not an agent yet. Lesson 3 adds the loop.

**How to run it in PyCharm:** click a code cell and press **Shift + Enter**. Run the cells in order from top to bottom.

## Step 1: Setup

This sets the model name and decides where the work log file will live.

In [19]:
import json
from datetime import datetime
from pathlib import Path

import ollama

MODEL = "qwen3:8b"

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

LOG_FILE = project_root / "data" / "work_log.jsonl"
LOG_FILE.parent.mkdir(exist_ok=True)

print("Log file:", LOG_FILE)

Log file: B:\_Dev\Python\Learn-AI-Agent\data\work_log.jsonl


### How to read it

One line showing the full path of the log file. It should end in `Learn-AI-Agent\data\work_log.jsonl`.

### What it means

The log will be a **JSON Lines** file (`.jsonl`): plain text with one entry per line, each line a small JSON record. It is easy to append to, easy to read back with Python or pandas, and you can open it in any text editor to check the agent's work.

The `data` folder is created if it does not exist. The file itself appears when the first entry is saved.

## Step 2: Write the tool as plain Python

A tool is an ordinary function. This one saves a work log entry. The cell also calls it once by hand, with no model involved.

In [20]:
def add_entry(project, description, hours=None):
    """Save one work log entry to the log file and return a short confirmation."""
    entry = {
        "logged_at": datetime.now().isoformat(timespec="seconds"),
        "project": project,
        "description": description,
        "hours": hours,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")
    return f"Saved entry for {project}."


print(add_entry("Learn-AI-Agent", "Tested add_entry by hand, no model involved", 0.1))
print()
print(LOG_FILE.read_text(encoding="utf-8"))

Saved entry for Learn-AI-Agent.

{"logged_at": "2026-11-11T19:55:52", "project": "Learn-AI-Agent", "description": "Tested add_entry by hand, no model involved", "hours": 0.1}
{"logged_at": "2026-11-11T19:56:16", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2}
{"logged_at": "2026-11-11T19:56:21", "project": "Herd Planner", "description": "Added vaccine records to herd planner", "hours": 1.5}
{"logged_at": "2026-11-11T19:56:29", "project": "Learn-AI-Agent", "description": "Ran lesson 2 of the agent notebook.", "hours": null}
{"logged_at": "2026-11-12T08:54:46", "project": "Learn-AI-Agent", "description": "Tested add_entry by hand, no model involved", "hours": 0.1}
{"logged_at": "2026-11-12T08:54:49", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2}
{"logged_at": "2026-11-12T08:54:53", "project": "Herd Planner", "description": "Added vaccine records to herd planner", "hours": 1.5}
{"logged_at": "2026-11-12T08:54:

### How to read it

First the confirmation text the function returns, then the contents of the log file: one line with `logged_at`, `project`, `description` and `hours`.

If you run this cell more than once, you will see one line per run. That is expected: the function appends.

### What it means

The tool works before the model is involved at all. That order matters. When an agent misbehaves, you want to know the tool itself is sound so you can look for the problem in the model's choices.

Two details that matter later:

- `hours=None` makes hours optional. The entry is still saved when the user did not say how long something took.
- The function **returns a short text result**. The model will read that text to learn whether the tool worked.

## Step 3: Describe the tool to the model

The model cannot see your Python code. It only sees a description: the tool's name, what it is for, and what inputs it takes.

In [21]:
ADD_ENTRY_TOOL = {
    "type": "function",
    "function": {
        "name": "add_entry",
        "description": "Save one work log entry. Use this when the user describes work they did.",
        "parameters": {
            "type": "object",
            "properties": {
                "project": {
                    "type": "string",
                    "description": "Name of the project the work was for.",
                },
                "description": {
                    "type": "string",
                    "description": "Short description of what was done.",
                },
                "hours": {
                    "type": "number",
                    "description": "Hours spent. Leave this out if the user did not say.",
                },
            },
            "required": ["project", "description"],
        },
    },
}

TOOLS = [ADD_ENTRY_TOOL]

print(json.dumps(ADD_ENTRY_TOOL, indent=2))

{
  "type": "function",
  "function": {
    "name": "add_entry",
    "description": "Save one work log entry. Use this when the user describes work they did.",
    "parameters": {
      "type": "object",
      "properties": {
        "project": {
          "type": "string",
          "description": "Name of the project the work was for."
        },
        "description": {
          "type": "string",
          "description": "Short description of what was done."
        },
        "hours": {
          "type": "number",
          "description": "Hours spent. Leave this out if the user did not say."
        }
      },
      "required": [
        "project",
        "description"
      ]
    }
  }
}


### How to read it

The same description printed back as indented JSON. Nothing has been sent to the model yet.

### What it means

This description is the whole of what the model knows about your tool:

- **name**: what the model writes when it wants the tool run. It must match the Python function's name.
- **description**: when to use it. This sentence does most of the work in getting the model to pick the tool at the right time.
- **parameters**: the inputs, each with a type and its own description.
- **required**: the inputs the model must always supply. `hours` is not in that list, so the model is allowed to leave it out.

Writing good tool descriptions is one of the main skills in building agents. When a model misuses a tool, the first fix is usually better wording here.

## Step 4: Ask the model, with the tool on offer

The same kind of call as lesson 1, with one addition: `tools=TOOLS`. The system message also gives the model your real project names.

In [22]:
KNOWN_PROJECTS = ["Cornerpost Watch", "Cornerpost Logic", "Herd Planner", "Learn-AI-Agent"]

SYSTEM_PROMPT = (
    "You are a work log assistant. "
    "When the user describes work they did, call the add_entry tool. "
    "Known projects: " + ", ".join(KNOWN_PROJECTS) + ". "
    "Use one of those names when the work matches one. "
    "Only include hours if the user stated them. "
    "If the message is not about work, answer normally without calling a tool."
)


def call_model(messages):
    """One call to the local model with the tools on offer."""
    return ollama.chat(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        think=False,
        options={"temperature": 0},
    )


entries_before = len(LOG_FILE.read_text(encoding="utf-8").splitlines())

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Spent 2 hours fixing the alert email bug in cornerpost watch."},
]

response = call_model(messages)

print("Text reply:    ", repr(response.message.content))
for call in response.message.tool_calls or []:
    print("Tool requested:", call.function.name)
    print("Arguments:     ", dict(call.function.arguments))

entries_after = len(LOG_FILE.read_text(encoding="utf-8").splitlines())
print()
print("Entries in the log file before the call:", entries_before)
print("Entries in the log file after the call: ", entries_after)

Text reply:     ''
Tool requested: add_entry
Arguments:      {'description': 'Fixed the alert email bug', 'hours': 2, 'project': 'Cornerpost Watch'}

Entries in the log file before the call: 9
Entries in the log file after the call:  9


### How to read it

- **Text reply** should be empty (`''`), or close to it.
- **Tool requested** should be `add_entry`.
- **Arguments** should show the project as `Cornerpost Watch` with proper capitals, a short description, and hours of `2`.
- The two entry counts should be **the same**.

If no tool was requested, run the cell again. If it keeps happening, tell me and we will adjust the wording.

### What it means

This is the central idea of the lesson. The model did not save anything, which is why the entry count did not change. It replied with a **request**: "please run `add_entry` with these inputs."

The model turned a loose sentence into structured data. You typed "cornerpost watch" in lowercase, and it matched that to the real project name from the list. It pulled out the hours as a number. That is the part ordinary code is bad at and a model is good at.

Compare this with lesson 1, where the model invented "Meat Price Data". Giving it the list of known projects is the fix.

## Step 5: Run the tool yourself

Your code looks up the function the model asked for and calls it with the model's arguments.

In [23]:
AVAILABLE_TOOLS = {"add_entry": add_entry}

if not response.message.tool_calls:
    print("The model did not ask for a tool. Run the Step 4 cell again first.")
else:
    call = response.message.tool_calls[0]
    tool_function = AVAILABLE_TOOLS[call.function.name]
    result = tool_function(**call.function.arguments)

    print("Tool result:", result)
    print()
    print("Last line of the log file:")
    print(LOG_FILE.read_text(encoding="utf-8").splitlines()[-1])

Tool result: Saved entry for Cornerpost Watch.

Last line of the log file:
{"logged_at": "2026-11-12T11:17:15", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2}


### How to read it

The tool's confirmation text, then the newest line in the log file. That line should hold the Cornerpost Watch entry with `"hours": 2`.

### What it means

Now the entry exists. Your code did the saving, using inputs the model chose.

`AVAILABLE_TOOLS` is a lookup table from tool name to Python function. It is also a safety boundary: the model can only cause functions in this table to run. If it asked for a tool called `delete_everything`, there would be nothing to call.

The `**` in `tool_function(**call.function.arguments)` unpacks the arguments, so `{"project": "X", "description": "Y"}` becomes `add_entry(project="X", description="Y")`.

## Step 6: Send the result back

The model asked for a tool but has not heard what happened. This adds two messages to the list, the model's request and the tool's result, and calls the model again.

In [24]:
messages.append(response.message)
messages.append({"role": "tool", "tool_name": call.function.name, "content": result})

final = call_model(messages)

print("Final reply:", final.message.content)
print()
print("The message list now holds:")
for m in messages:
    role = m["role"] if isinstance(m, dict) else m.role
    print("  -", role)

Final reply: The entry for fixing the alert email bug in Cornerpost Watch has been saved.

The message list now holds:
  - system
  - user
  - assistant
  - tool


### How to read it

A short confirmation in plain English, such as a sentence saying the entry was logged. Below it, the four roles in the list: `system`, `user`, `assistant`, `tool`.

### What it means

There is a new role: **`tool`**. It carries the result of running a tool back to the model.

The full round trip is:

1. `user`: what you typed
2. `assistant`: the model's request to run `add_entry`
3. `tool`: what `add_entry` returned
4. `assistant`: the model's final answer, written after reading the result

This is lesson 1's idea again: the message list is the memory. The model only knows the tool ran because the result is in the list.

## Step 7: The model decides when not to use the tool

Same setup, but the message is not about work.

In [25]:
other = call_model([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is the capital of Colorado?"},
])

print("Tool requested:", bool(other.message.tool_calls))
print("Text reply:    ", other.message.content)

Tool requested: False
Text reply:     The capital of Colorado is Denver.


### How to read it

**Tool requested** should be `False`, and the text reply should answer the question (Denver).

### What it means

Offering a tool does not force the model to use it. On every call it chooses between answering in text and asking for a tool. That choice is the "one model decision" in level 2, and it is the decision an agent makes over and over.

## Step 8: Wrap the round trip in one function

Steps 4 to 6 in a single function, then three test sentences.

In [26]:
def run_one_turn(user_text):
    """Handle one user message: let the model ask for tools once, run them, return its final reply."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    response = call_model(messages)

    if not response.message.tool_calls:
        return response.message.content

    messages.append(response.message)
    for call in response.message.tool_calls:
        name = call.function.name
        arguments = dict(call.function.arguments)
        if name not in AVAILABLE_TOOLS:
            result = f"Error: there is no tool named {name}."
        else:
            try:
                result = AVAILABLE_TOOLS[name](**arguments)
            except TypeError as error:
                result = f"Error: {error}"
        print(f"  [tool] {name}({arguments}) -> {result}")
        messages.append({"role": "tool", "tool_name": name, "content": result})

    final = call_model(messages)
    return final.message.content


for text in [
    "Added vaccine records to herd planner, took about an hour and a half.",
    "Ran lesson 2 of the agent notebook.",
    "How many days are in a leap year?",
]:
    print("You:  ", text)
    print("Model:", run_one_turn(text))
    print()

You:   Added vaccine records to herd planner, took about an hour and a half.
  [tool] add_entry({'description': 'Added vaccine records to herd planner', 'hours': 1.5, 'project': 'Herd Planner'}) -> Saved entry for Herd Planner.
Model: The entry for Herd Planner has been successfully saved. Let me know if you need anything else!

You:   Ran lesson 2 of the agent notebook.
  [tool] add_entry({'description': 'Ran lesson 2 of the agent notebook.', 'project': 'Learn-AI-Agent'}) -> Saved entry for Learn-AI-Agent.
Model: The entry for "Ran lesson 2 of the agent notebook" has been successfully saved under the Learn-AI-Agent project. Let me know if you'd like to log anything else!

You:   How many days are in a leap year?
Model: A leap year has 366 days.



### How to read it

For each sentence you see what you typed, a `[tool]` line if a tool was run (with the arguments and the result), and the model's final reply.

- The first should log to `Herd Planner` with hours of `1.5`.
- The second should log to `Learn-AI-Agent`. Look at the hours: you did not give any, so the best outcome is no hours at all.
- The third should have no `[tool]` line and a plain answer.

### What it means

`run_one_turn` is a complete, useful program: type a sentence, get a saved record. For a job this simple, it is all you need, and it is not an agent. The steps are fixed: ask once, run the tools, ask once more.

The `try` and `except` matter. If the model sends an input the function does not accept, the code does not crash. It sends the error text back as the tool result, and the model can read it and explain what went wrong.

Check the second sentence closely. If the model filled in hours that you never stated, that is the "inventing missing information" weakness from lesson 1's exercise. The next step measures how often that happens.

## Step 9: Measure it

A small test set with the right answer for each sentence. This step only looks at what the model asks for. It does not run the tool, so nothing is added to your log.

In [27]:
TEST_CASES = [
    {"text": "Fixed the alert email bug in Cornerpost Watch for about 2 hours.",
     "tool": True, "project": "Cornerpost Watch", "hours": 2},
    {"text": "Spent an hour on herd planner vaccine records.",
     "tool": True, "project": "Herd Planner", "hours": 1},
    {"text": "Ran lesson 1 of the agent notebook.",
     "tool": True, "project": "Learn-AI-Agent", "hours": None},
    {"text": "Worked on the cornerpost logic company site this morning.",
     "tool": True, "project": "Cornerpost Logic", "hours": None},
    {"text": "What is the capital of Colorado?",
     "tool": False, "project": None, "hours": None},
    {"text": "Remind me what a JSON file is.",
     "tool": False, "project": None, "hours": None},
]


def same_hours(got, expected):
    if expected is None:
        return got is None
    try:
        return float(got) == float(expected)
    except (TypeError, ValueError):
        return False


scores = {"tool": 0, "project": 0, "hours": 0}

for case in TEST_CASES:
    response = call_model([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": case["text"]},
    ])
    calls = response.message.tool_calls or []
    arguments = dict(calls[0].function.arguments) if calls else {}

    tool_ok = bool(calls) == case["tool"]
    project_ok = arguments.get("project") == case["project"]
    hours_ok = same_hours(arguments.get("hours"), case["hours"])

    scores["tool"] += tool_ok
    scores["project"] += project_ok
    scores["hours"] += hours_ok

    print(case["text"])
    print(f"   tool decision: {'right' if tool_ok else 'WRONG'}"
          f"   project: {'right' if project_ok else 'WRONG'} ({arguments.get('project')})"
          f"   hours: {'right' if hours_ok else 'WRONG'} ({arguments.get('hours')})")

total = len(TEST_CASES)
print()
print(f"Tool decision right: {scores['tool']} of {total}")
print(f"Project right:       {scores['project']} of {total}")
print(f"Hours right:         {scores['hours']} of {total}")

Fixed the alert email bug in Cornerpost Watch for about 2 hours.
   tool decision: right   project: right (Cornerpost Watch)   hours: right (2)
Spent an hour on herd planner vaccine records.
   tool decision: right   project: right (Herd Planner)   hours: right (1)
Ran lesson 1 of the agent notebook.
   tool decision: right   project: right (Learn-AI-Agent)   hours: right (None)
Worked on the cornerpost logic company site this morning.
   tool decision: right   project: right (Cornerpost Logic)   hours: right (None)
What is the capital of Colorado?
   tool decision: right   project: right (None)   hours: right (None)
Remind me what a JSON file is.
   tool decision: right   project: right (None)   hours: right (None)

Tool decision right: 6 of 6
Project right:       6 of 6
Hours right:         6 of 6


### How to read it

For each test sentence, three checks:

- **tool decision**: did the model ask for the tool when it should, and stay quiet when it should not?
- **project**: did it pick the exact expected project name? The value it gave is in brackets.
- **hours**: did it give the stated hours, and leave hours out when none were stated?

The three totals at the bottom are the scores out of 6.

### What it means

This is an **evaluation**, the same idea as scoring a model on a test set in data science. It replaces "it seemed to work when I tried it" with a number.

It also tells you where to look when something is wrong:

- A wrong **tool decision** points to the tool's description or the system prompt's rule about when to log.
- A wrong **project** points to the known projects list or how the sentence names the project.
- Wrong **hours** almost always means the model invented a number. The fix is firmer wording, or checking in code.

Six cases is a tiny test set, so one mistake swings the score a lot. The habit matters more than the number: every time you change the prompt or the tool description, rerun this cell and see whether the scores went up or down.

## Try it yourself

Change one thing at a time, then rerun Step 9 to see the effect on the scores:

1. Edit `KNOWN_PROJECTS` in Step 4 to your real list of projects, then rerun Steps 4 to 9. Add a test case for one of the new names.
2. In Step 4, delete the sentence `"Only include hours if the user stated them. "` from the system prompt. Does the hours score drop?
3. In Step 3, change the tool's description to something vague, such as `"A tool."`. Does the tool decision score drop?
4. Add two test cases of your own to `TEST_CASES`, one that should be logged and one that should not.

Exercises 2 and 3 show how much the wording controls the model's behavior.

## What it all means

- A tool is an ordinary Python function plus a description the model can read.
- The model never runs a tool. It asks, your code runs it, and your code sends the result back as a `tool` message.
- The table of available tools is the limit on what the model can cause to happen.
- A small test set turns "does it work?" into scores you can track.

## What is next and why

**Lesson 3: The loop.** Right now the model gets one chance to ask for tools. That breaks on a request like "log an hour on Herd Planner, then tell me what I have logged today", which needs two tools used one after the other, with the second depending on the first.

Lesson 3 adds a second tool, `search_entries`, and replaces the fixed steps with a loop that keeps going until the model stops asking for tools. That loop is what makes it an agent.